## Module B — Statistics, hypothesis tests, and tail dependence (daily workflow)\n
\n
This notebook reproduces the B-module analysis on **daily data**:\n
\n
- Descriptive stats (mean/std/skew/kurtosis)\n
- Correlation (Pearson + Spearman)\n
- OLS regression with **robust** standard errors (t-tests in summary)\n
- Tail subset (worst 5% NFLX days) vs full sample\n
- Copula comparison (Gaussian vs Student-t; Archimedean when valid)\n
\n
Outputs are also written to `outputs/` when you run the script `b_module_stats_tail.py`.\n

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.stats as st
import statsmodels.api as sm
from statsmodels.distributions.copula.api import (
    ClaytonCopula,
    GaussianCopula,
    GumbelCopula,
    StudentTCopula,
)

ROOT = Path.cwd()
# VS Code sometimes sets the notebook working directory to `notebooks/`.
# If we don't see the data files here, step up one level.
if not (ROOT / "NFLX.csv").exists():
    ROOT = ROOT.parent

OUT_DIR = ROOT / "outputs"
OUT_DIR.mkdir(exist_ok=True)

TAIL_Q = 0.05


In [2]:
def rank_to_uniform(x: pd.Series) -> np.ndarray:
    r = st.rankdata(x.to_numpy(), method="average")
    n = len(r)
    return (r - 0.5) / n


def load_clean_nflx(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.dropna(subset=["Date"]).sort_values("Date")
    df = df[["Date", "Adj Close"]].rename(columns={"Adj Close": "NFLX_AdjClose"})
    df["NFLX_AdjClose"] = pd.to_numeric(df["NFLX_AdjClose"], errors="coerce")
    return df.dropna(subset=["NFLX_AdjClose"])


def load_clean_gld(path: Path) -> pd.DataFrame:
    last_err = None
    for enc in ("utf-8-sig", "cp1252", "latin1"):
        try:
            df = pd.read_csv(path, skiprows=3, encoding=enc)
            last_err = None
            break
        except Exception as e:
            last_err = e
    if last_err is not None:
        raise last_err

    df["Date"] = pd.to_datetime(df["Date"], format="%d-%b-%Y", errors="coerce")
    df = df.dropna(subset=["Date"]).sort_values("Date")
    df = df[["Date", "NAV"]].rename(columns={"NAV": "GLD_NAV"})
    df["GLD_NAV"] = pd.to_numeric(df["GLD_NAV"], errors="coerce")
    return df.dropna(subset=["GLD_NAV"])


def add_log_returns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["NFLX_logret"] = np.log(out["NFLX_AdjClose"]).diff()
    out["GLD_logret"] = np.log(out["GLD_NAV"]).diff()
    return out.dropna(subset=["NFLX_logret", "GLD_logret"]).reset_index(drop=True)


In [3]:
nflx = load_clean_nflx(ROOT / "NFLX.csv")
gld = load_clean_gld(ROOT / "navhist-us-en-gld(navhist).csv")
merged = pd.merge(nflx, gld, on="Date", how="inner").sort_values("Date").reset_index(drop=True)
df = add_log_returns(merged)

merged.shape, df.shape


((4793, 3), (4792, 5))

### Descriptive statistics\n

In [4]:
cols = ["NFLX_AdjClose", "GLD_NAV", "NFLX_logret", "GLD_logret"]

desc = df[cols].describe().T

desc["skew"] = df[cols].skew(numeric_only=True)

desc["kurtosis_excess"] = df[cols].kurtosis(numeric_only=True)

desc


SyntaxError: unexpected character after line continuation character (1787541970.py, line 1)

### Correlation (returns)\n

In [ ]:
pearson = st.pearsonr(df["NFLX_logret"], df["GLD_logret"])

spearman = st.spearmanr(df["NFLX_logret"], df["GLD_logret"])



{

    "pearson_r": pearson.statistic,

    "pearson_p": pearson.pvalue,

    "spearman_rho": spearman.statistic,

    "spearman_p": spearman.pvalue,

}


### OLS regression (robust SE)\n
\n
Model: `NFLX_logret ~ 1 + GLD_logret`\n

In [ ]:
y = df["NFLX_logret"]

X = sm.add_constant(df["GLD_logret"])

res = sm.OLS(y, X, missing="drop").fit(cov_type="HC1")

print(res.summary())


### Tail subset (worst 5% NFLX days)\n

In [ ]:
cutoff = df["NFLX_logret"].quantile(TAIL_Q)

tail = df[df["NFLX_logret"] <= cutoff].copy()

len(tail), cutoff


In [ ]:
pearson_t = st.pearsonr(tail["NFLX_logret"], tail["GLD_logret"])

spearman_t = st.spearmanr(tail["NFLX_logret"], tail["GLD_logret"])



{

    "tail_pearson_r": pearson_t.statistic,

    "tail_pearson_p": pearson_t.pvalue,

    "tail_spearman_rho": spearman_t.statistic,

    "tail_spearman_p": spearman_t.pvalue,

}


### Copula comparison (empirical CDF transform)\n
\n
- Gaussian copula (no tail dependence)\n
- Student-t copula (symmetric tail dependence)\n
- Clayton/Gumbel only when parameter range is valid\n

In [ ]:
def copula_aic_table(x: pd.Series, y: pd.Series) -> pd.DataFrame:

    u = rank_to_uniform(x)

    v = rank_to_uniform(y)

    data_uv = np.column_stack([u, v])



    def safe_ll(cop, *args):

        dens = cop.pdf(data_uv, *args)

        dens = np.clip(dens, 1e-300, np.inf)

        return float(np.sum(np.log(dens)))



    rows = []



    # Gaussian

    gc = GaussianCopula()

    gc.fit_corr_param(data_uv)

    ll = safe_ll(gc)

    rho = float(np.asarray(gc.corr)[0, 1])

    rows.append({"copula": "gaussian", "params": {"rho": rho}, "loglik": ll, "k": 1})



    # t copula: search df grid

    best = None

    for df_ in [3, 4, 5, 7, 10, 15, 20, 30]:

        tc = StudentTCopula(df=df_)

        tc.fit_corr_param(data_uv)

        ll_ = safe_ll(tc)

        rho_ = float(np.asarray(tc.corr)[0, 1])

        cand = (ll_, df_, rho_)

        if best is None or cand[0] > best[0]:

            best = cand

    ll, df_best, rho_best = best

    rows.append({"copula": "t", "params": {"rho": rho_best, "df": int(df_best)}, "loglik": ll, "k": 2})



    # Archimedean by tau sign

    tau = st.kendalltau(u, v).statistic

    if tau is not None and tau > 0:

        clay = ClaytonCopula()

        theta = float(clay.fit_corr_param(data_uv))

        if theta > 0:

            rows.append({"copula": "clayton", "params": {"theta": theta}, "loglik": safe_ll(clay, theta), "k": 1})

        gum = GumbelCopula()

        theta = float(gum.fit_corr_param(data_uv))

        if theta >= 1:

            rows.append({"copula": "gumbel", "params": {"theta": theta}, "loglik": safe_ll(gum, theta), "k": 1})



    out = pd.DataFrame(rows)

    out["aic"] = 2 * out["k"] - 2 * out["loglik"]

    out["n"] = len(u)

    return out.sort_values("aic").reset_index(drop=True)





cop_full = copula_aic_table(df["NFLX_logret"], df["GLD_logret"])

cop_tail = copula_aic_table(tail["NFLX_logret"], tail["GLD_logret"])



cop_full, cop_tail
